In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from huggingface_hub import HfFileSystem
import polars as pl

fs = HfFileSystem()

number_of_files = 1
event_type = 'ttbar_pu0'
number_of_hf_repo_files= 100
# Load particles
particles_list = []
for i in range(number_of_files):
    file_path = f"datasets/CERN/ColliderML-Release-1/data/{event_type}_particles/train-{i:05d}-of-{number_of_hf_repo_files:05d}.parquet"
    with fs.open(file_path, "rb") as f:
        particles_list.append(pl.read_parquet(f))
particles = pl.concat(particles_list)

# Load calo_hits
calo_hits_list = []
for i in range(number_of_files):
    file_path = f"datasets/CERN/ColliderML-Release-1/data/{event_type}_calo_hits/train-{i:05d}-of-{number_of_hf_repo_files:05d}.parquet"
    with fs.open(file_path, "rb") as f:
        calo_hits_list.append(pl.read_parquet(f))
calo_hits = pl.concat(calo_hits_list)

# Load tracks
tracks_list = []
for i in range(number_of_files):
    file_path = f"datasets/CERN/ColliderML-Release-1/data/{event_type}_tracks/train-{i:05d}-of-{number_of_hf_repo_files:05d}.parquet"
    with fs.open(file_path, "rb") as f:
        tracks_list.append(pl.read_parquet(f))
tracks = pl.concat(tracks_list)

In [3]:
(particles.lazy()
 .select(['event_id', 'particle_id', 'pdg_id', 'vx', 'vy', 'vz'])
 .explode('particle_id', 'pdg_id', 'vx', 'vy', 'vz')
.filter((pl.col('event_id')==999) & (pl.col('particle_id')==726))   
 ).collect()

event_id,particle_id,pdg_id,vx,vy,vz
u32,u64,i64,f32,f32,f32
999,726,111,-0.002103,0.020381,-23.405081


In [5]:
(calo_hits.lazy()
    .select(['event_id', 'contrib_particle_ids','contrib_energies', 'x', 'y', 'z','detector'])
    .explode('contrib_particle_ids', 'contrib_energies', 'x', 'y', 'z', 'detector')
    .explode('contrib_particle_ids', 'contrib_energies') # Double explode if list[list]
    .rename({'contrib_particle_ids': 'particle_id', 'contrib_energies': 'energy_contribution'})
    .filter((pl.col('event_id')==999) & (pl.col('particle_id')==726))).collect()

event_id,particle_id,energy_contribution,x,y,z,detector
u32,u64,f32,f32,f32,f32,u8
999,726,0.001105,754.233643,406.257233,-3207.449951,9


In [6]:
(particles.lazy()
 .select(['event_id','particle_id', 'parent_id', 'pdg_id'])
 .explode('particle_id','pdg_id', 'parent_id')
    .filter((pl.col('parent_id')==726) & (pl.col('event_id')==999))
).collect()

event_id,particle_id,parent_id,pdg_id
u32,u64,i64,i64
999,4486,726,22
